<a href="https://colab.research.google.com/github/Lisura123/J26-IT-348/blob/component2-tinyml/component2-tinyml/notebooks/colab_pruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Component 2: Pruning feasibility study E1 (Colab)

Runs the pruning experiment fixed in `docs/PRUNING.md` (`prune-e1-v1`) **before** any result is seen.

- **No retraining and no fine-tuning.** The frozen Float32 model (`models/`) and the frozen INT8 parameters (`models/int8/`) are only read; pruned variants exist in memory and in the report.
- Variants: unstructured magnitude pruning at 10 / 20 / 30 % of the 208 weights; structured removal of 2 / 3 / 5 hidden units (experimental only: breaks the 229-value FL layout).
- **TRAINING rows only.** OPERATION rows are dropped straight after loading and are never used.
- Size and multiply-add numbers are **estimates, not hardware measurements**. No ESP32 measurement is made.

All code comes from `src/tinyml/prune.py` and `scripts/08_pruning_study.py`. This notebook only calls it.

**Two manual safety settings:**
1. **Step 1:** paste the full 40-character approved commit hash into `APPROVED_COMMIT`. The notebook checks out exactly that commit (never a newer one) and checks that it belongs to `component2-tinyml`.
2. **Step 5:** the study runs only after you change `CONFIRM_PRUNING = False` to `True` by hand. Step 5 also re-checks the commit.

Run the cells one at a time with **Shift + Enter** (not "Run all").

## Step 1: Get the approved code revision and install dependencies
Paste the **full 40-character commit hash** you were given for the official run into `APPROVED_COMMIT`, then run the cell. The cell checks out exactly that commit and stops if the hash is missing, malformed, unknown or not part of `component2-tinyml`. It never moves to a newer branch revision.

In [1]:
# STEP 1: Check out the approved revision and install dependencies
import os, re, sys, subprocess

APPROVED_COMMIT = "4f9effeb121fbf86abd8e4eb2de2735457007eea"   # paste the full 40-character approved commit hash here

REPO_URL = "https://github.com/Lisura123/J26-IT-348"
BRANCH = "component2-tinyml"
REPO_DIR = "/content/J26-IT-348"
COMPONENT_DIR = f"{REPO_DIR}/component2-tinyml"

if not re.fullmatch(r"[0-9a-f]{40}", APPROVED_COMMIT):
    raise SystemExit("Paste the full 40-character approved commit hash into APPROVED_COMMIT first.")

def git(*args, check=True):
    return subprocess.run(["git", "-C", REPO_DIR, *args], capture_output=True, text=True, check=check)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
git("fetch", "origin", BRANCH)
if git("cat-file", "-e", f"{APPROVED_COMMIT}^{{commit}}", check=False).returncode != 0:
    raise SystemExit(f"Commit {APPROVED_COMMIT} was not found in the repository.")
if git("merge-base", "--is-ancestor", APPROVED_COMMIT, f"origin/{BRANCH}", check=False).returncode != 0:
    raise SystemExit(f"Commit {APPROVED_COMMIT} is not part of origin/{BRANCH}.")
git("checkout", "--quiet", "--detach", APPROVED_COMMIT)          # exactly this revision, never a newer one
HEAD = git("rev-parse", "HEAD").stdout.strip()
if HEAD != APPROVED_COMMIT or git("status", "--porcelain").stdout.strip():
    raise SystemExit(f"Checked-out revision {HEAD} is not the clean approved revision {APPROVED_COMMIT}.")
COMMIT = HEAD[:7]
print("Approved revision checked out:", git("log", "-1", "--format=%H %s").stdout.strip())
newer = git("rev-list", "--count", f"{APPROVED_COMMIT}..origin/{BRANCH}").stdout.strip()
print(f"Newer commits on origin/{BRANCH} (not used): {newer}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{COMPONENT_DIR}/requirements.txt"], check=True)
import numpy as np, pandas as pd, sklearn
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__)

Approved revision checked out: 4f9effeb121fbf86abd8e4eb2de2735457007eea Add Pruning Feasibility Study E1 Framework
Newer commits on origin/component2-tinyml (not used): 0
Python 3.13.16 | numpy 2.1.3 | pandas 2.2.3 | scikit-learn 1.6.1


## Step 2: Import the modules and verify the frozen Float32 and INT8 models (no data is read)

In [2]:
# STEP 2: Import modules and verify the committed model artifacts
import importlib.util
SRC_DIR = f"{COMPONENT_DIR}/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from tinyml.validate import file_sha256
from tinyml.train import check_dataset_sha256
from tinyml.holdout import verify_frozen_model
from tinyml.prune import STUDY_VERSION

MODEL_DIR = f"{COMPONENT_DIR}/models"
contract = verify_frozen_model(MODEL_DIR)          # pinned SHA-256 values; stops on any difference
spec = importlib.util.spec_from_file_location("prune_study", f"{COMPONENT_DIR}/scripts/08_pruning_study.py")
prune_study = importlib.util.module_from_spec(spec); spec.loader.exec_module(prune_study)
q = prune_study.load_frozen_int8(f"{MODEL_DIR}/int8")   # pinned INT8 SHA-256 values
print("Study       :", STUDY_VERSION)
print("Model       :", contract["model_version"], "| weights", contract["weights_sha256"][:16], "...")
print("INT8        :", q.version, "| M1", q.m1, "| SH1", q.sh1)
print("Frozen Float32 and INT8 models verified.")

Study       : prune-e1-v1
Model       : c2-local-v1 | weights 94e7ab9384a1906e ...
INT8        : int8-v1 | M1 2113405489 | SH1 38
Frozen Float32 and INT8 models verified.


## Step 3: Upload the dataset
Click **Choose Files** and select `pv_fault_dataset.csv`. Only the TRAINING rows are used.

In [3]:
# STEP 3: Upload pv_fault_dataset.csv
from google.colab import files
uploaded = files.upload()
csv_files = [n for n in uploaded if n.endswith(".csv")]
if not csv_files:
    raise SystemExit("No CSV uploaded. Run this cell again.")
DATA_PATH = os.path.abspath(csv_files[0])
print("Dataset file:", DATA_PATH, "|", os.path.getsize(DATA_PATH), "bytes")

Saving pv_fault_dataset.csv to pv_fault_dataset.csv
Dataset file: /content/pv_fault_dataset.csv | 957366 bytes


## Step 4: Check the dataset fingerprint (stops if it is not the validated file)

In [4]:
# STEP 4: SHA-256 check
sha = check_dataset_sha256(file_sha256(DATA_PATH))
print("SHA-256 OK:", sha)

SHA-256 OK: 5fbb93471c056c274cbf81e8c317934e379b2484013147e4c2cacfa93421595b


## Step 5: Run the pruning study (TRAINING rows only, once)
Derives the pruned variants in memory, compares them with the frozen Float32 and INT8 models on status-OK TRAINING rows, and writes the report to `/content/prune_out/results/`. It refuses to overwrite earlier output and writes nothing into `models/`.

**This cell refuses to run until you change `CONFIRM_PRUNING = False` to `True` by hand.** It also re-checks that the approved commit from Step 1 is still checked out. Run it **once**.

In [5]:
# STEP 5: Pruning study E1 (no OPERATION data, no retraining) - official run, once
CONFIRM_PRUNING = True   # change to True by hand for the official run

if not CONFIRM_PRUNING:
    raise SystemExit("Set CONFIRM_PRUNING = True in this cell to run the official pruning study.")
head_now = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                          capture_output=True, text=True, check=True).stdout.strip()
if head_now != APPROVED_COMMIT:
    raise SystemExit(f"Checked-out revision {head_now} is not the approved {APPROVED_COMMIT}. Re-run Step 1.")

SPLITS_PATH = f"{COMPONENT_DIR}/results/splits_runwise_v1.json"
OUT_ROOT = "/content/prune_out"
report = prune_study.run(DATA_PATH, SPLITS_PATH, MODEL_DIR, OUT_ROOT, code_commit=COMMIT)
print("TRAINING rows:", report["training_rows"], "| status-OK compared:", report["status_ok_rows"],
      "| OPERATION used:", report["operation_window_used"])
for v in report["variants"]:
    print(f"{v['name']:20s} float32 {'PASS' if v['e1_checks']['float32']['passed'] else 'FAIL'}"
          f" | int8 {'PASS' if v['e1_checks']['int8']['passed'] else 'FAIL'}")
print(report["estimate_note"])

TRAINING rows: 3000 | status-OK compared: 2924 | OPERATION used: False
baseline             float32 PASS | int8 PASS
unstructured_10pct   float32 PASS | int8 PASS
unstructured_20pct   float32 PASS | int8 PASS
unstructured_30pct   float32 PASS | int8 PASS
structured_minus2    float32 PASS | int8 PASS
structured_minus3    float32 PASS | int8 PASS
structured_minus5    float32 PASS | int8 PASS
ESTIMATE, not a hardware measurement: analytical parameter-storage bytes and multiply-add counts. Sparse formats exclude decoding code; no ESP32 flash, RAM, latency or energy was measured.


## Step 6: Show the report

In [6]:
# STEP 6: Show the Markdown report
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT_ROOT}/results/pruning_study.md", encoding="utf-8").read()))

# Pruning feasibility study E1 (TRAINING window only)

Study `prune-e1-v1`, created 2026-10-16T20:39:55Z. Model `c2-local-v1`, weights SHA-256 `94e7ab9384a1906ec2cf0d3176b435e700f15cd99552fbc6c71519c648ecc3a7`, INT8 `int8-v1`. Code `4f9effe`. Environment {'python': '3.13.16', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1'}.

Rows: 3000 TRAINING, 2924 status-OK compared, 76 skipped by the input gates. **OPERATION was not used. No retraining or fine-tuning. No model artifact was written.**

Frozen INT8 vs frozen Float32 agreement: 1.0000.

## Prediction agreement with the frozen Float32 model

| variant | hidden | non-zero W (f32 / int8) | f32 agree | f32 changes | f32 OPEN/SHORT | int8 agree | int8 changes | int8 OPEN/SHORT | int8 vs frozen int8 | E1 f32 | E1 int8 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 16 | 208 / 206 | 1.0000 | 0 | 0 | 1.0000 | 0 | 0 | 1.0000 | PASS | PASS |
| unstructured_10pct | 16 | 187 / 187 | 1.0000 | 0 | 0 | 1.0000 | 0 | 0 | 1.0000 | PASS | PASS |
| unstructured_20pct | 16 | 166 / 166 | 1.0000 | 0 | 0 | 1.0000 | 0 | 0 | 1.0000 | PASS | PASS |
| unstructured_30pct | 16 | 146 / 146 | 1.0000 | 0 | 0 | 1.0000 | 0 | 0 | 1.0000 | PASS | PASS |
| structured_minus2 | 14 | 182 / 180 | 1.0000 | 0 | 0 | 1.0000 | 0 | 0 | 1.0000 | PASS | PASS |
| structured_minus3 | 13 | 169 / 167 | 1.0000 | 0 | 0 | 1.0000 | 0 | 0 | 1.0000 | PASS | PASS |
| structured_minus5 | 11 | 143 / 142 | 1.0000 | 0 | 0 | 1.0000 | 0 | 0 | 1.0000 | PASS | PASS |

## Training-fit macro-F1 (frozen Float32 vs variant; not independent accuracy)

| variant | frozen f32 | pruned f32 | drop | pruned int8 | drop |
|---|---|---|---|---|---|
| baseline | 1.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 |
| unstructured_10pct | 1.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 |
| unstructured_20pct | 1.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 |
| unstructured_30pct | 1.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 |
| structured_minus2 | 1.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 |
| structured_minus3 | 1.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 |
| structured_minus5 | 1.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 |

## Changed OPEN/SHORT predictions and change pairs

- baseline (pruned_float_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- baseline (pruned_int8_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- unstructured_10pct (pruned_float_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- unstructured_10pct (pruned_int8_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- unstructured_20pct (pruned_float_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- unstructured_20pct (pruned_int8_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- unstructured_30pct (pruned_float_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- unstructured_30pct (pruned_int8_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- structured_minus2 (pruned_float_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- structured_minus2 (pruned_int8_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- structured_minus3 (pruned_float_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- structured_minus3 (pruned_int8_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- structured_minus5 (pruned_float_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none
- structured_minus5 (pruned_int8_vs_frozen_float): OPEN/SHORT {'OPEN_CIRCUIT': {'changed_to': 0, 'changed_from': 0}, 'SHORT_CIRCUIT': {'changed_to': 0, 'changed_from': 0}}; pairs none

## Size and compute ESTIMATES (not hardware measurements)

_ESTIMATE, not a hardware measurement: analytical parameter-storage bytes and multiply-add counts. Sparse formats exclude decoding code; no ESP32 flash, RAM, latency or energy was measured._

| variant | stored params (dense) | f32 dense B | f32 bitmap B | f32 index B | int8 dense B | int8 bitmap B | int8 index B | MACs dense | MACs zero-skip (int8) | FL layout |
|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 229 | 916 | 942 | 1124 | 561 | 585 | 765 | 208 | 206 | unchanged (229 values) |
| unstructured_10pct | 229 | 916 | 858 | 1019 | 561 | 566 | 727 | 208 | 187 | unchanged shape (229 values); FL would need a shared fixed mask |
| unstructured_20pct | 229 | 916 | 774 | 914 | 561 | 545 | 685 | 208 | 166 | unchanged shape (229 values); FL would need a shared fixed mask |
| unstructured_30pct | 229 | 916 | 694 | 814 | 561 | 525 | 645 | 208 | 146 | unchanged shape (229 values); FL would need a shared fixed mask |
| structured_minus2 | 201 | 804 | 827 | 986 | 527 | 548 | 705 | 182 | 180 | BROKEN: 8-14-5 has 201 values, not 229 (experimental only) |
| structured_minus3 | 187 | 748 | 770 | 917 | 510 | 530 | 675 | 169 | 167 | BROKEN: 8-13-5 has 187 values, not 229 (experimental only) |
| structured_minus5 | 159 | 636 | 654 | 779 | 476 | 493 | 617 | 143 | 142 | BROKEN: 8-11-5 has 159 values, not 229 (experimental only) |

INT8 bytes include the 256-byte tanh table and 13 bytes of constants; biases are int32 / float32.

## Conclusion rule

E1 checks 1-2 only. Adoption also needs a measured on-device benefit (criterion 3) and an unchanged FL layout (criterion 4); neither is decided by E1.

## Caveats

- All comparisons use TRAINING rows, on which the model was fitted; agreement and training-fit macro-F1 are not independent accuracy.
- OPERATION was not used and is not used to choose a pruning level.
- Storage and multiply-add figures are estimates; no ESP32 measurement was made.
- Structured variants change the architecture and break the 229-value FL layout (experimental only).
- Prediction agreement does not establish electrical safety; OPEN/SHORT outputs stay unconfirmed ML predictions.


## Step 7: Download the results
Downloads `prune_out.zip` (study report in JSON and Markdown). **Send it back for review.** Nothing is committed until it is reviewed and approved.

In [7]:
# STEP 7: Zip and download
import shutil
zip_path = shutil.make_archive("/content/prune_out", "zip", OUT_ROOT)
files.download(zip_path)
print("Downloaded", zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded /content/prune_out.zip
